# Linger Workflow
Below is the file structure. We start with fresh `LINGER_output/` and `data/` folders.
```
.
├── LINGER_data/
│   └── data_bulk/
│
├── LINGER_output/
│   ├── cell_population_TF_RE_binding.txt
│   ├── cell_population_cis_regulatory.txt
│   ├── cell_population_trans_regulatory.txt
│   └── ...
│
└── data/
    ├── Peaks.txt
    ├── TG_pseudobulk.tsv
    ├── RE_pseudobulk.tsv
    ├── adata_RNA.h5ad
    └── adata_ATAC.h5ad
```

In [61]:
rm -rf LINGER_output/*

In [62]:
rm -rf data/*

In [16]:
from joblib import Parallel, delayed
import time

# Function to simulate work
def square(x):
    import time
    time.sleep(0.5)  # simulate some computation
    return x * x

nums = list(range(10))

# --- Sequential execution ---
start = time.time()
results_seq = [square(n) for n in nums]
end = time.time()
print("Sequential results:", results_seq)
print("Sequential time: {:.2f} s".format(end - start))

# --- Parallel execution with joblib ---
start = time.time()
results_par = Parallel(n_jobs=4)(delayed(square)(n) for n in nums)
end = time.time()
print("Parallel results:  ", results_par)
print("Parallel time: {:.2f} s".format(end - start))

Sequential results: [0, 1, 4, 9, 16, 25, 36, 49, 64, 81]
Sequential time: 5.01 s
Parallel results:   [0, 1, 4, 9, 16, 25, 36, 49, 64, 81]
Parallel time: 1.51 s


## 1. Get the input data
- `.h5` matrix with both RNA and ATAC
- cell type annotation

In [63]:
%%bash
wget --progress=bar:force -O data/pbmc_granulocyte_sorted_10k_filtered_feature_bc_matrix.h5 https://cf.10xgenomics.com/samples/cell-arc/1.0.0/pbmc_granulocyte_sorted_10k/pbmc_granulocyte_sorted_10k_filtered_feature_bc_matrix.h5
wget --progress=bar:force -O data --load-cookies /tmp/cookies.txt "https://docs.google.com/uc?export=download&confirm=$(wget --quiet --save-cookies /tmp/cookies.txt --keep-session-cookies --no-check-certificate 'https://docs.google.com/uc?export=download&id=17PXkQJr8fk0h90dCkTi3RGPmFNtDqHO_' -O- | sed -rn 's/.*confirm=([0-9A-Za-z_]+).*/\1\n/p')&id=17PXkQJr8fk0h90dCkTi3RGPmFNtDqHO_" -O data/PBMC_label.txt && rm -rf /tmp/cookies.txt

--2026-03-06 12:24:32--  https://cf.10xgenomics.com/samples/cell-arc/1.0.0/pbmc_granulocyte_sorted_10k/pbmc_granulocyte_sorted_10k_filtered_feature_bc_matrix.h5
Resolving cf.10xgenomics.com (cf.10xgenomics.com)... 104.18.0.173, 104.18.1.173, 2606:4700::6812:1ad, ...
Connecting to cf.10xgenomics.com (cf.10xgenomics.com)|104.18.0.173|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 162282142 (155M) [binary/octet-stream]
Saving to: 'data/pbmc_granulocyte_sorted_10k_filtered_feature_bc_matrix.h5'

data/pbmc_granulocy 100%[===================>] 154.76M   227MB/s    in 0.7s    

2026-03-06 12:24:33 (227 MB/s) - 'data/pbmc_granulocyte_sorted_10k_filtered_feature_bc_matrix.h5' saved [162282142/162282142]

--2026-03-06 12:24:34--  https://docs.google.com/uc?export=download&confirm=&id=17PXkQJr8fk0h90dCkTi3RGPmFNtDqHO_
Resolving docs.google.com (docs.google.com)... 173.194.76.100, 173.194.76.101, 173.194.76.102, ...
Connecting to docs.google.com (docs.google.com)|173.194

In [64]:
ls data

PBMC_label.txt  pbmc_granulocyte_sorted_10k_filtered_feature_bc_matrix.h5


In [65]:
import scanpy as sc
adata = sc.read_10x_h5('data/pbmc_granulocyte_sorted_10k_filtered_feature_bc_matrix.h5', gex_only=False)

Variable names are not unique. To make them unique, call `.var_names_make_unique`.


In [66]:
import scipy.sparse as sp
import pandas as pd
matrix=adata.X.T
adata.var['gene_ids']=adata.var.index
features=pd.DataFrame(adata.var['gene_ids'].values.tolist(),columns=[1])
features[2]=adata.var['feature_types'].values
barcodes=pd.DataFrame(adata.obs_names,columns=[0])
label = pd.read_csv('data/PBMC_label.txt',sep='\t',header=0)
from LingerGRN.preprocess import *
adata_RNA,adata_ATAC=get_adata(matrix,features,barcodes,label)# adata_RNA and adata_ATAC are scRNA and scATAC

Trying to modify attribute `.obs` of view, initializing view as actual.
Trying to modify attribute `.obs` of view, initializing view as actual.


In [67]:
adata_RNA

View of AnnData object with n_obs × n_vars = 9543 × 36601
    obs: 'barcode', 'sample', 'label', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt'
    var: 'gene_ids', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'

In [68]:
adata_ATAC

View of AnnData object with n_obs × n_vars = 9543 × 108377
    obs: 'barcode', 'sample', 'label'
    var: 'gene_ids'

## 2. Filter cells and genes

In [69]:
sc.pp.filter_cells(adata_RNA, min_genes=200)
sc.pp.filter_genes(adata_RNA, min_cells=3)
sc.pp.filter_cells(adata_ATAC, min_genes=200)
sc.pp.filter_genes(adata_ATAC, min_cells=3)

selected_barcode = list(set(adata_RNA.obs['barcode'].values) & set(adata_ATAC.obs['barcode'].values))

barcode_idx = pd.DataFrame(range(adata_RNA.shape[0]), index=adata_RNA.obs['barcode'].values)
adata_RNA = adata_RNA[barcode_idx.loc[selected_barcode][0]]

barcode_idx = pd.DataFrame(range(adata_ATAC.shape[0]), index=adata_ATAC.obs['barcode'].values)
adata_ATAC = adata_ATAC[barcode_idx.loc[selected_barcode][0]]

Trying to modify attribute `.obs` of view, initializing view as actual.
Trying to modify attribute `.obs` of view, initializing view as actual.


## 3 Pseudo-bulk (`pseudo_bulk.py`)
This function creates *metacells* : aggregated pseudo-cells that represent local neighborhoods in the data. 

In [71]:
# Generate pseudo-bulk/metacell
import os
from LingerGRN.pseudo_bulk import *

samplelist=list(set(adata_ATAC.obs['sample'].values)) # sample is generated from cell barcode 
tempsample=samplelist[0]

TG_pseudobulk=pd.DataFrame([])
RE_pseudobulk=pd.DataFrame([])

n_samples = adata_RNA.obs['sample'].nunique()
singlepseudobulk = (n_samples > 10)

for tempsample in samplelist:

    # get cells from only tempsample
    adata_RNAtemp = adata_RNA[adata_RNA.obs['sample' ] == tempsample]
    adata_ATACtemp = adata_ATAC[adata_ATAC.obs['sample'] == tempsample]

    TG_pseudobulk_temp, RE_pseudobulk_temp = pseudo_bulk(adata_RNAtemp, adata_ATACtemp, singlepseudobulk)  
    
    TG_pseudobulk = pd.concat([TG_pseudobulk, TG_pseudobulk_temp], axis=1)
    RE_pseudobulk = pd.concat([RE_pseudobulk, RE_pseudobulk_temp], axis=1)
    
    RE_pseudobulk[RE_pseudobulk > 100] = 100

Received a view of an AnnData. Making a copy.
Received a view of an AnnData. Making a copy.
Received a view of an AnnData. Making a copy.
Received a view of an AnnData. Making a copy.


In [72]:
if not os.path.exists('data/'):
    os.mkdir('data/')
    
adata_ATAC.write('data/adata_ATAC.h5ad')
adata_RNA.write('data/adata_RNA.h5ad')
TG_pseudobulk=TG_pseudobulk.fillna(0)
RE_pseudobulk=RE_pseudobulk.fillna(0)
pd.DataFrame(adata_ATAC.var['gene_ids']).to_csv('data/Peaks.txt',header=None,index=None)
TG_pseudobulk.to_csv('data/TG_pseudobulk.tsv')
RE_pseudobulk.to_csv('data/RE_pseudobulk.tsv')

Trying to modify attribute `.obs` of view, initializing view as actual.
Trying to modify attribute `.obs` of view, initializing view as actual.


In [73]:
ls data

PBMC_label.txt     adata_ATAC.h5ad
Peaks.txt          adata_RNA.h5ad
RE_pseudobulk.tsv  pbmc_granulocyte_sorted_10k_filtered_feature_bc_matrix.h5
TG_pseudobulk.tsv


## 4. Preprocess

In [74]:
import os
from LingerGRN.preprocess import *
source = "/globalsc/ucl/inma/vangysel/Linger/"
GRNdir =  source + "LINGER_data/data_bulk/"
genome = 'hg38'
outdir = source + "LINGER_output/" 

In [75]:
method = 'LINGER'  

In [76]:
preprocess(TG_pseudobulk, RE_pseudobulk, GRNdir, genome, method, outdir)

Mapping gene expression...
Generate TF expression...
Generate RE chromatin accessibility...
Generate TF binding...


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████| 23/23 [04:43<00:00, 12.34s/it]


Generate Index...


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████| 14907/14907 [00:43<00:00, 339.42it/s]


### About the preprocess function

1) `extract_overlap_regions` : overlap our peaks with the reference peaks
2) `gene_expression` : filters `TG_pseudobulk` to genes present in `bulk_gene_all.txt`, applies log2(1+x) and writes :
   - Exp.txt — log2 normalized expression matrix
   - Symbol.txt — gene names
   - Col.txt — metacell column names
<br><br>
3) `TF_expression` : subsets `Exp.txt` to TF genes only, writes:
    - TFexp.txt — TF expression matrix
    - TFName.txt — TF names
<br><br>
4) `RE_pseudobulk → Openness.txt` : peaks are written in the file without transformation
<br><br>
6) `load_TFbinding`: reads `MotifTarget_Matrix_{chr}.txt` and `MotifTarget_hg19_hg38_{chr}.txt` for all chromosomes, maps motif binding scores to your peaks, normalizes by motif weight, aggregates motifs to TFs via Match2.txt, writes:
    - TF_binding.txt — peaks × TFs motif binding matrix where each value represents how strongly each TF's motif is present at each peak
<br><br>
7) `index_generate` loop : for each gene in `Symbol.txt`, records which RE indices and TF indices are relevant for that gene's neural network input (which TFs and which REs to use as input features for that gene) writes:
    - index.txt — per-gene mapping of RE and TF indices

```
extract_overlap_regions
  → Region.bed
  → match_hg19_peak.bed
  → hg19_Peak_hg19_gene_u.txt
  → MotifTarget_hg19_hg38_{chr}.txt  (×23)
  → Region_overlap_{chr}.bed         (×23)

gene_expression
  → Exp.txt
  → Symbol.txt
  → Col.txt

TF_expression
  → TFexp.txt
  → TFName.txt

directly
  → Openness.txt

load_TFbinding
  → TF_binding.txt

index_generate
  → index.txt

```

In [83]:
import pandas as pd
import torch

outdir  = 'LINGER_output/'
GRNdir  = 'LINGER_data/data_bulk/'

# --- preprocess outputs ---
Exp        = pd.read_csv(outdir + 'Exp.txt',         sep='\t', header=None)
Symbol     = pd.read_csv(outdir + 'Symbol.txt',      sep='\t', header=None)
Col        = pd.read_csv(outdir + 'Col.txt',         sep='\t', header=None)
TFexp      = pd.read_csv(outdir + 'TFexp.txt',       sep='\t', header=None)
TFName     = pd.read_csv(outdir + 'TFName.txt',      sep='\t', header=None)
Openness   = pd.read_csv(outdir + 'Openness.txt',    sep='\t', header=None)
TF_binding = pd.read_csv(outdir + 'TF_binding.txt',  sep='\t', header=None)
index      = pd.read_csv(outdir + 'index.txt',       sep='\t', header=None)
Region     = pd.read_csv(outdir + 'Region.bed',      sep='\t', header=None)
Region_overlap_chr1 = pd.read_csv(outdir + 'Region_overlap_chr1.bed', sep='\t', header=None)
hg19_Peak  = pd.read_csv(outdir + 'hg19_Peak_hg19_gene_u.txt', sep='\t', header=None)
MotifTarget_chr1 = pd.read_csv(outdir + 'MotifTarget_hg19_hg38_chr1.txt', sep='\t', header=None)
match_hg19 = pd.read_csv(outdir + 'match_hg19_peak.bed', sep='\t', header=None)
print()

In [98]:
ls LINGER_output

Col.txt                          Region_overlap_chr11.bed
Exp.txt                          Region_overlap_chr12.bed
MotifTarget_hg19_hg38_chr1.txt   Region_overlap_chr13.bed
MotifTarget_hg19_hg38_chr10.txt  Region_overlap_chr14.bed
MotifTarget_hg19_hg38_chr11.txt  Region_overlap_chr15.bed
MotifTarget_hg19_hg38_chr12.txt  Region_overlap_chr16.bed
MotifTarget_hg19_hg38_chr13.txt  Region_overlap_chr17.bed
MotifTarget_hg19_hg38_chr14.txt  Region_overlap_chr18.bed
MotifTarget_hg19_hg38_chr15.txt  Region_overlap_chr19.bed
MotifTarget_hg19_hg38_chr16.txt  Region_overlap_chr2.bed
MotifTarget_hg19_hg38_chr17.txt  Region_overlap_chr20.bed
MotifTarget_hg19_hg38_chr18.txt  Region_overlap_chr21.bed
MotifTarget_hg19_hg38_chr19.txt  Region_overlap_chr22.bed
MotifTarget_hg19_hg38_chr2.txt   Region_overlap_chr3.bed
MotifTarget_hg19_hg38_chr20.txt  Region_overlap_chr4.bed
MotifTarget_hg19_hg38_chr21.txt  Region_overlap_chr5.bed
MotifTarget_hg19_hg38_chr22.txt  Region_overlap_chr6.bed
MotifTarget_hg19_hg

## 5. Training


For each gene, LINGER trains a small neural network that predicts that gene's expression across metacells from its neighboring TFs and nearby peaks. The bulk atlas provides a starting point via pretrained weights and Fisher information, and the network is fine-tuned on your pseudobulk data.

### 5.1 Load_data
Loads everything preprocess produced plus bulk atlas indices. Creates :
- `data_merge.txt` : merge of Symbol.txt with gene_all, creates a table that maps each of your genes to its bulk atlas index and chromosome. This is how LINGER knows which pretrained bulk model corresponds to each gene in your dataset.
- `TF_match` : a mapping between your TF indices and bulk TF indices, needed to slice the right columns from the pretrained weight matrix.

``` 
outdir/Exp.txt          → Target  (genes × metacells, what the network predicts)
outdir/TFexp.txt        → Exp     (TFs × metacells, TF input features)
outdir/Openness.txt     → Opn     (peaks × metacells, RE input features)
outdir/TF_binding.txt   → adj_matrix_all  (peaks × TFs, motif binding — graph regularizer)
outdir/index.txt        → idx     (per-gene RE/TF indices)
GRNdir/{chr}_gene.txt   → gene_all  (bulk gene list with bulk indices)
GRNdir/{chr}_index.txt  → idx_bulk  (bulk per-gene TF/RE indices)
GRNdir/TFName.txt       → TFName_b  (bulk TF names)
outdir/TFName.txt       → TFName_s  (your TF names)
```

### 5.2 Chromosome loop

For each of the 23 chr, locate the genes on chr and load its bulk fisher information and bulk pre-trained weights (in `GRNdir`).

Then `sc_nn` is called for each chr. 

- `all_models_{chr}.pt` contains one pretrained network per gene in the bulk atlas. 

- `fisher_{chr}.pt` contains the Fisher information matrix for each bulk network — a measure of which weights were most important for the bulk task, used to prevent catastrophic forgetting.

### 5.3 `sc_nn` : per gene network training

Get inputs for NN of gene `ii` :

```python
TFtemp = Exp[TFidxtemp, :]               # expression of relevant TFs across metacells
REtemp = Opn[REidxtemp, :]               # accessibility of nearby peaks across metacells
inputs = np.vstack((TFtemp, REtemp))     # stack → (TFs+REs) × metacells
```

Also builds the **Graph Laplacian regularizer** that encodes the motif binding structure as a graph — TFs connected to the peaks they bind. Used in the loss to encourage the network to respect known TF-RE binding relationships <br>

Train for 100 epoch with loss being :

```python
loss = mse_loss(y_pred, y_tr)           # fit your pseudobulk expression
    + l1_norm * l1_lambda               # L1 regularization (sparsity)
    + l2_bulk                           # pull weights toward bulk pretrained values
    + lap_reg                           # respect TF-RE graph structure
```

### 5.4 After training
SHAP values are computed for every input feature (TFs and REs) across all metacells. These quantify how much each TF and each RE contributed to predicting this gene's expression — they become the regulatory scores used by `cis_reg` (RE-TG) and `trans_reg` (TF-TG) later.

```python
background = X_tr[np.random.choice(X_tr.shape[0], 50, replace=False)]
explainer = shap.DeepExplainer(net,background)
shap_values = explainer.shap_values(X_tr)
```

- net_{chr}.pt    — trained networks, one per gene (used by TF_RE_binding in LINGER mode)
- shap_{chr}.pt   — SHAP values, one per gene    (used by cis_reg and trans_reg)
- result_{chr}.txt — training scores per gene
- Loss_{chr}.txt  — loss curves per gene

### 5.6 Compelete training call stack
```
training(method='LINGER')
  ├── load_data()
  │     ← Exp.txt, TFexp.txt, Openness.txt, TF_binding.txt
  │     ← index.txt, Symbol.txt
  │     ← GRNdir/{chr}_gene.txt, {chr}_index.txt, {chr}_index_all.txt
  │     ← GRNdir/TFName.txt
  │     → data_merge.txt
  └── for each chr:
        ├── load fisher_{chr}.pt, all_models_{chr}.pt  (bulk prior)
        └── for each gene on this chr:
              sc_nn(...)
                ├── slice TF + RE inputs from Exp.txt + Openness.txt
                ├── build graph Laplacian from TF_binding.txt
                ├── align with bulk model weights via TF_match
                ├── train 100 epochs with MSE + L1 + EWC + Laplacian loss
                └── compute SHAP values
              → net_{chr}.pt
              → shap_{chr}.pt
              → result_{chr}.txt
              → Loss_{chr}.txt
```

In [33]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn import functional as F
from scipy.stats import pearsonr
from scipy.stats import spearmanr
#load data
import numpy as np
import pandas as pd
import random
from torch.optim import Adam
import os
from sklearn.linear_model import ElasticNet
from sklearn.datasets import make_regression
from sklearn.model_selection import KFold
import shap
hidden_size  = 64
hidden_size2 = 16
output_size = 1
from joblib import Parallel,delayed
seed_value = 42

class Net(nn.Module):
    def __init__(self,input_size,activef):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(input_size, 64)
        self.fc2 = nn.Linear(64, 16)
        self.fc3 = nn.Linear(16, output_size)
        self.activef=activef
    def forward(self, x):
        #x = torch.sigmoid(self.fc1(x))
        if self.activef=='ReLU':
            x = F.relu(self.fc1(x))
            x = F.relu(self.fc2(x))
        if self.activef=='sigmoid':
            x = F.sigmoid(self.fc1(x))
            x = F.sigmoid(self.fc2(x))
        if self.activef=='tanh':
            x = F.tanh(self.fc1(x))
            x = F.tanh(self.fc2(x))
        x = self.fc3(x)
        return x

#EWC
def EWC(fisher,params,net):
    params_n = list(net.parameters())
    EWC=0
    i=0 
    p=params_n[0]
    cost=(p-params[i])*fisher*(p-params[i])
    EWC=EWC+cost.sum()
    return EWC


def sc_nn(ii,gene_chr,TFindex,TFindex_bulk,REindex,REindex_bulk,REindex_bulk_match,Target,netall,adj_matrix_all,Exp,TF_match,input_size_all,fisherall,Opn,l1_lambda,fisher_w,activef):
    warnings.filterwarnings("ignore")
    alpha = 1
    eps=1e-12
    alpha = torch.tensor(alpha,dtype=torch.float32)
    gene_idx=gene_chr['id_s'].values[ii]-1
    gene_idx_b=int(gene_chr['id_b'].values[ii])-1
    TFidxtemp=TFindex[gene_idx]
    TFidxtemp=TFidxtemp.split('_')
    TFidxtemp=[int(TFidxtemp[k])+1 for k in range(len(TFidxtemp))]
    TFidxtemp_b=TFindex_bulk[gene_idx_b]
    TFidxtemp_b=TFidxtemp_b.split('_')
    TFidxtemp_b=[int(TFidxtemp_b[k]) for k in range(len(TFidxtemp_b))]
    TFtemp=Exp[np.array(TFidxtemp)-1,:]
    REidxtemp=REindex[gene_idx]
    REidxtemp_b_m=REindex_bulk_match[gene_idx]
    REidxtemp_b=REindex_bulk[gene_idx_b]
    REidxtemp=str(REidxtemp).split('_')
    REidxtemp_b_m=str(REidxtemp_b_m).split('_')
    REidxtemp_b=str(REidxtemp_b).split('_')
    if (len(REidxtemp)==1)&(REidxtemp[0]=='nan'):
        REidxtemp=[]
        REidxtemp_b_m=[]
        inputs=TFtemp+1-1
        L=np.zeros([len(TFidxtemp)+len(REidxtemp),len(TFidxtemp)+len(REidxtemp)])
        L=torch.tensor(L, dtype=torch.float32)
    else:
        REidxtemp=[int(REidxtemp[k])+1 for k in range(len(REidxtemp))]
        REidxtemp_b_m=[int(REidxtemp_b_m[k])+1 for k in range(len(REidxtemp_b_m))]
        REtemp=Opn[np.array(REidxtemp)-1,:]
        inputs=np.vstack((TFtemp, REtemp))
        adj_matrix=np.zeros([len(TFidxtemp)+len(REidxtemp),len(TFidxtemp)+len(REidxtemp)])
        AA=adj_matrix_all[np.array(REidxtemp)-1,:]
        AA=AA[:,np.array(TFidxtemp)-1]
        adj_matrix[:len(TFidxtemp),-len(REidxtemp):]=AA.T
        adj_matrix[-len(REidxtemp):,:len(TFidxtemp)]=AA
        A = torch.tensor(adj_matrix, dtype=torch.float32)
        D = torch.diag(A.sum(1))
        degree = A.sum(dim=1)
        degree += eps
        D_sqrt_inv = 1 / degree.sqrt()
        D_sqrt_inv = torch.diag(D_sqrt_inv)
        L = D_sqrt_inv@(D - A)@D_sqrt_inv
    if (len(REidxtemp_b)==1)&(REidxtemp_b[0]=='nan'):
        REidxtemp_b=[]
    else:
        REidxtemp_b=[int(REidxtemp_b[k]) for k in range(len(REidxtemp_b))]
    targets = torch.tensor(Target[gene_idx,:])
    inputs = torch.tensor(inputs,dtype=torch.float32)
    targets = targets.type(torch.float32)
    mean = inputs.mean(dim=1)
    std = inputs.std(dim=1)
    inputs = (inputs.T - mean) / (std+eps)
    inputs=inputs.T
    num_nodes=inputs.shape[0]
    y=targets.reshape(len(targets),1)     
    #trainData testData          
    input_size=int(input_size_all[gene_idx_b])
    loaded_net = Net(input_size,activef)
    loaded_net.load_state_dict(netall[gene_idx_b])
    params = list(loaded_net.parameters())
    fisher0=fisherall[gene_idx_b][0].data.clone()
    data0=pd.DataFrame(TFidxtemp)
    data1=pd.DataFrame(TFidxtemp_b)
    data0.columns=['TF']
    data1.columns=['TF']
    A=TF_match.loc[data0['TF'].values-1]['id_b']
    data0=pd.DataFrame(A)
    data0.columns=['TF']
    data1['id_b']=data1.index
    data0['id_s']=range(0,len(A))
    merge_TF=pd.merge(data0,data1,how='left',on='TF')
    if (len(REidxtemp)>0)&(len(REidxtemp_b)>0):
        data0=pd.DataFrame(REidxtemp_b_m)
        data1=pd.DataFrame(REidxtemp_b)
        data0.columns=['RE']
        data1.columns=['RE']
        data0['id_s']=data0.index
        data1['id_b']=data1.index
        merge_RE=pd.merge(data0,data1,how='left',on='RE')
        if merge_RE['id_b'].isna().sum()==0:
            good=1
            indexall=merge_TF['id_b'].values.tolist()+(merge_RE['id_b'].values+merge_TF.shape[0]).tolist()
        else: 
            good=0
    else:
        indexall=merge_TF['id_b'].values.tolist()
        good=1
    if good==1:      
        fisher=fisher0[:,np.array(indexall,dtype=int)]
        params_bulk = params[0][:,np.array(indexall,dtype=int)]
        with torch.no_grad():
            params_bulk = params_bulk.detach()     
        num_nodes=inputs.shape[0]
        n_folds = 5
        kf = KFold(n_splits=n_folds,shuffle=True,random_state=0)
        fold_size = len(inputs.T) // n_folds
        input_size = num_nodes
        mse_loss = nn.MSELoss()
        y_pred_all=0*(y+1-1)
        y_pred_all1=0*(y+1-1)
        y_pred_all1=y_pred_all1.numpy().reshape(-1)
        X_tr = inputs.T
        y_tr = y
        torch.manual_seed(seed_value)
        net = Net(input_size,activef)
        optimizer = Adam(net.parameters(),lr=0.01,weight_decay=l1_lambda)   
            #optimizer = Adam(net.parameters(),weight_decay=1)
            # Perform backpropagation
        Loss0=np.zeros([100,1])
        for i in range(100):
            # Perform forward pass
            y_pred = net(X_tr)
            # Calculate loss
            l1_norm = sum(torch.linalg.norm(p, 1) for p in net.parameters())
            #loss_EWC=EWC(fisher,params_bulk,net);
            l2_bulk = -1* fisher_w*  sum(sum(torch.mul(params_bulk,net.fc1.weight)))
            lap_reg = alpha * torch.trace(torch.mm(torch.mm(net.fc1.weight, L), net.fc1.weight.t()))
            loss = mse_loss(y_pred, y_tr) +l1_norm*l1_lambda+l2_bulk+lap_reg 
            Loss0[i,0]=loss.detach().numpy()
            # Perform backpropagation
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        np.random.seed(42)
        background = X_tr[np.random.choice(X_tr.shape[0], 50, replace=False)]
        explainer = shap.DeepExplainer(net,background)
        shap_values = explainer.shap_values(X_tr)
        warnings.resetwarnings()
        return net,shap_values,0.5,0.5,1,Loss0
    else:
        warnings.resetwarnings()
        return 0,0,0,0,0,0

def get_TSS(GRNdir,genome,TSS_dis):
    #import pyensembl
# Initialize Ensembl database for the desired genome assembly
    #ensembl = pyensembl.EnsemblRelease(release=release, species=species)  # For hg19
# ensembl = pyensembl.EnsemblRelease(release=104, species='mouse')  # For mm10
# Get all genes in the genome
    #genes = ensembl.genes()
# Retrieve TSS positions for each gene and store them in a list
    #tss_positions = []
    #strand=[]
    #chrom=[]
    #genesymbol=[]
    #for gene in genes:
        #tss_positions.append(gene.transcripts[0].start)
        #strand.append(gene.strand)
        #chrom.append('chr'+gene.contig)
        #genesymbol.append(gene.name)
    import pandas as pd
    Tssdf = pd.read_csv(GRNdir+'TSS_'+genome+'.txt',sep='\t',header=None)
    Tssdf.columns=['chr','TSS','symbol','strand']
    Tssdf['1M-']=Tssdf['TSS']-TSS_dis
    Tssdf['1M+']=Tssdf['TSS']+TSS_dis
    temp=Tssdf['1M-'].values
    temp[temp<1]=1
    Tssdf['1M-']=temp
    Tssdf=Tssdf[Tssdf['symbol']!='']
    Tssdf[['chr','1M-','1M+','symbol','TSS', 'strand']].to_csv('data/TSS_extend_1M.txt',sep='\t',index=None)

def load_data(GRNdir,outdir):
    gene_all=pd.DataFrame([])
    for i in range(22):
        chr='chr'+str(i+1)
        gene_file=GRNdir+chr+'_gene.txt'
        data0=pd.read_csv(gene_file,sep='\t',header=None)
        data0['chr']=chr
        data0['id_b']=data0.index+1
        gene_all=pd.concat([gene_all,data0])
    chr='chrX'
    gene_file=GRNdir+chr+'_gene.txt'
    data0=pd.read_csv(gene_file,sep='\t',header=None)
    data0['chr']=chr
    data0['id_b']=data0.index+1
    gene_all=pd.concat([gene_all,data0])
    gene_file=outdir+'Symbol.txt'
    data0=pd.read_csv(gene_file,sep='\t',header=None)
    data0.columns=['Symbol']
    data0['id_s']=data0.index+1
    gene_all.columns=['Symbol','chr','id_b']
    data_merge=pd.merge(data0,gene_all,how='left',on='Symbol')
    TFName_b=pd.read_csv(GRNdir+'TFName.txt',header=None,sep='\t')
    TFName_s=pd.read_csv(outdir+'TFName.txt',header=None,sep='\t')
    TFName_b.columns=['TF']
    TFName_s.columns=['TF']
    TFName_b['id_b']=TFName_b.index+1# index from 1
    TFName_s['id_s']=TFName_s.index+1# index from 1
    TF_match=pd.merge(TFName_s,TFName_b,how='left',on='TF')
    Opn_file=outdir+'Openness.txt'
    idx_file=outdir+'index.txt'
    geneexp_file=outdir+'Exp.txt'
    Target=pd.read_csv(geneexp_file,header=None,sep='\t')
    Target=Target.values
    #def sc_NN(gene_file,Opn_file,idx_file,geneexp_file,out_PCC,out_net):
    #alpha = torch.tensor(alpha,dtype=torch.float32)
    bind_file=outdir+'TF_binding.txt'
    adj_matrix_all=pd.read_csv(bind_file,header=None,sep='\t')
    adj_matrix_all=adj_matrix_all.values
    TFExp_file=outdir+'TFexp.txt'
    Opn=pd.read_csv(Opn_file,header=None,sep='\t')
    Opn=Opn.values
    idx=pd.read_csv(idx_file,header=None,sep='\t')
    Exp=pd.read_csv(TFExp_file,header=None,sep='\t')
    Exp=Exp.values
    return Exp,idx,Opn,adj_matrix_all,Target,data_merge,TF_match
def sc_nn_NN(ii,RE_TGlink_temp,Target,Exp,Opn,l1_lambda,activef):
    warnings.filterwarnings("ignore")
    alpha = 1
    eps=1e-12
    alpha = torch.tensor(alpha,dtype=torch.float32)
    if RE_TGlink_temp[0] in Exp.index:
        TFtemp = Exp.drop([RE_TGlink_temp[0]]).values
    else:
        TFtemp=Exp.values
    REtemp=Opn.loc[RE_TGlink_temp[1]].values
    inputs=np.vstack((TFtemp, REtemp))
    targets = torch.tensor(Target.loc[RE_TGlink_temp[0],:])
    inputs = torch.tensor(inputs,dtype=torch.float32)
    targets = targets.type(torch.float32)
    mean = inputs.mean(dim=1)
    std = inputs.std(dim=1)
    inputs = (inputs.T - mean) / (std+eps)
    inputs=inputs.T
    num_nodes=inputs.shape[0]
    y=targets.reshape(len(targets),1)     
    #trainData testData          
    input_size=int(num_nodes)
    mse_loss = nn.MSELoss()
    y_pred_all=0*(y+1-1)
    y_pred_all1=0*(y+1-1)
    y_pred_all1=y_pred_all1.numpy().reshape(-1)
    X_tr = inputs.T
    y_tr = y
    torch.manual_seed(seed_value)
    net = Net(input_size,activef)
    optimizer = Adam(net.parameters(),lr=0.01,weight_decay=l1_lambda)   
            #optimizer = Adam(net.parameters(),weight_decay=1)
            # Perform backpropagation
    Loss0=np.zeros([100,1])
    for i in range(100):
            # Perform forward pass
        y_pred = net(X_tr)
            # Calculate loss
        l1_norm = sum(torch.linalg.norm(p, 1) for p in net.parameters())
            #loss_EWC=EWC(fisher,params_bulk,net);
        #l2_bulk = -1* fisher_w*  sum(sum(torch.mul(params_bulk,net.fc1.weight)))
        #lap_reg = alpha * torch.trace(torch.mm(torch.mm(net.fc1.weight, L), net.fc1.weight.t()))
        loss = mse_loss(y_pred, y_tr) +l1_norm*l1_lambda#+l2_bulk+lap_reg 
        Loss0[i,0]=loss.detach().numpy()
            # Perform backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    np.random.seed(42)
    background = X_tr[np.random.choice(X_tr.shape[0], 50, replace=False)]
    explainer = shap.DeepExplainer(net,background)
    shap_values = explainer.shap_values(X_tr)
    warnings.resetwarnings()
    return net,shap_values,Loss0
    

def load_data_scNN(GRNdir,species):
    import pandas as pd
    if species=='New':
        Match2=pd.read_csv(GRNdir+'MotifMatch.txt',header=0,sep='\t')
    else:
        Match2=pd.read_csv(GRNdir+'Match_TF_motif_'+species+'.txt',header=None,sep='\t')
        Match2.columns = ['Motif','TF']
    TFName = pd.DataFrame(Match2['TF'].unique())
    Target=pd.read_csv('data/TG_pseudobulk.tsv',sep=',',header=0,index_col=0)
    TFlist=list(set(Target.index)&set(TFName[0].values))
    Exp=Target.loc[TFlist]
    Opn=pd.read_csv('data/RE_pseudobulk.tsv',sep=',',header=0,index_col=0)
    RE_TGlink=pd.read_csv('data/RE_gene_distance.txt',sep='\t',header=0)
    RE_TGlink = RE_TGlink.groupby('gene').apply(lambda x: x['RE'].values.tolist()).reset_index()
    geneoverlap=list(set(Target.index)&set(RE_TGlink['gene']))
    RE_TGlink.index=RE_TGlink['gene']
    RE_TGlink=RE_TGlink.loc[geneoverlap]
    RE_TGlink=RE_TGlink.reset_index(drop=True)
    return Exp,Opn,Target,RE_TGlink  

def RE_TG_dis(outdir):
    import pandas as pd
    import pybedtools
    import numpy as np
    print('Overlap the regions with gene loc ...')
    import os# Create the directory
    current_directory = os.getcwd()
    os.makedirs(outdir, exist_ok=True)
    import pandas as pd
    peakList=pd.read_csv(current_directory+'/data/Peaks.txt',index_col=None,header=None)
    peakList1=[temp.split(':')[0] for temp in peakList[0].values.tolist()]
    peakList2=[temp.split(':')[1].split('-')[0] for temp in peakList[0].values.tolist()]
    peakList3=[temp.split(':')[1].split('-')[1] for temp in peakList[0].values.tolist()]
    peakList['chr']=peakList1
    peakList['start']=peakList2
    peakList['end']=peakList3
    peakList[['chr','start','end']].to_csv(current_directory+'/data/Peaks.bed',sep='\t',header=None,index=None)
    TSS_1M=pd.read_csv(current_directory+'/data/TSS_extend_1M.txt',sep='\t',header=0)
    TSS_1M.to_csv(current_directory+'/data/TSS_extend_1M.bed',sep='\t',header=None,index=None)
    a = pybedtools.example_bedtool(current_directory+'/data/Peaks.bed')
    b = pybedtools.example_bedtool(current_directory+'/data/TSS_extend_1M.bed')
    a_with_b = a.intersect(b, wa=True,wb=True)
    a_with_b.saveas(outdir+'temp.bed')
    a_with_b=pd.read_csv(outdir+'temp.bed',sep='\t',header=None)
    a_with_b['RE']=a_with_b[0].astype(str) + ':' + a_with_b[1].astype(str) + '-' + a_with_b[2].astype(str)
    temp=a_with_b[['RE',6]]
    temp.columns=[['RE','gene']]
    temp['distance']=np.abs(a_with_b[7]-a_with_b[1])
    temp.to_csv(current_directory+'/data/RE_gene_distance.txt',sep='\t',index=None)
    

def process_chr(chr, GRNdir, outdir, data_merge, idx, Target, adj_matrix_all, Exp, TF_match, Opn, l1_lambda, fisher_w, activef):

    netall_s={}
    shapall_s={}
    result=np.zeros([data_merge.shape[0],2])
    Lossall=np.zeros([data_merge.shape[0],100])

    torch.set_num_threads(1)

    #print(chr, flush=True)

    idx_file1=GRNdir+chr+'_index.txt'
    idx_file_all=GRNdir+chr+'_index_all.txt'

    idx_bulk=pd.read_csv(idx_file1,header=None,sep='\t') 
    idxRE_all=pd.read_csv(idx_file_all,header=None,sep='\t')

    gene_chr=data_merge[data_merge['chr']==chr]
    N=len(gene_chr)

    TFindex=idx.values[:,2]
    REindex=idx.values[:,1]
    REindex_bulk_match=idx.values[:,3]
    REindex_bulk=idxRE_all.values[:,0]
    TFindex_bulk=idx_bulk.values[:,2]
    input_size_all=idx_bulk.values[:,3]

    fisherall = torch.load(GRNdir+'fisher_'+chr+'.pt')
    netall=torch.load(GRNdir+'all_models_'+chr+'.pt')

    for ii in range(N):
        warnings.filterwarnings("ignore")

        res=sc_nn(
            ii,gene_chr,TFindex,TFindex_bulk,REindex,REindex_bulk,
            REindex_bulk_match,Target,netall,adj_matrix_all,Exp,
            TF_match,input_size_all,fisherall,Opn,l1_lambda,
            fisher_w,activef
        )

        warnings.resetwarnings()

        index_all=gene_chr.index[ii]

        if res[4]==1:
            result[index_all,0]=res[2]
            result[index_all,1]=res[3]
            netall_s[index_all]=res[0]
            shapall_s[index_all]=res[1]
            Lossall[index_all,:]=res[5].T
        else:
            result[index_all,0]=-100

    result=pd.DataFrame(result)
    result.index=data_merge['Symbol'].values

    genetemp=data_merge[data_merge['chr']==chr]['Symbol'].values
    result=result.loc[genetemp]

    result.to_csv(outdir+'result_'+chr+'.txt',sep='\t')
    torch.save(netall_s,outdir+'net_'+chr+'.pt')
    torch.save(shapall_s,outdir+'shap_'+chr+'.pt')

    Lossall=pd.DataFrame(Lossall)
    Lossall.index=data_merge['Symbol'].values
    Lossall=Lossall.loc[genetemp]
    Lossall.to_csv(outdir+'Loss_'+chr+'.txt',sep='\t')


from tqdm import tqdm
import warnings
import time
import pandas as pd
import numpy as np
def training(GRNdir,method,outdir,activef,species):
    if method=='LINGER':
        hidden_size  = 64
        hidden_size2 = 16
        output_size = 1
        l1_lambda = 0.01 
        alpha_l = 0.01#elastic net parameter
        lambda0 = 0.00 #bulk
        fisher_w=0.1
        n_jobs=16

        Exp,idx,Opn,adj_matrix_all,Target,data_merge,TF_match=load_data(GRNdir,outdir)
        data_merge.to_csv(outdir+'data_merge.txt',sep='\t')
        chrall=['chr'+str(i+1) for i in range(22)]
        chrall.append('chrX')
        import warnings
        import time
        from tqdm import tqdm
        
        ### parellelize for loop 
        from joblib import Parallel, delayed
        Parallel(n_jobs=8,  backend='loky', verbose=10)(
            delayed(process_chr)(
                chr, GRNdir, outdir, data_merge, idx, Target,
                adj_matrix_all, Exp, TF_match, Opn,
                l1_lambda, fisher_w, activef
            )
            for chr in chrall
        )       
        #### parellelize for loop 

    if method=='scNN':
        hidden_size  = 64
        hidden_size2 = 16
        output_size = 1
        l1_lambda = 0.01 
        alpha_l = 0.01#elastic net parameter
        lambda0 = 0.00 #bulk
        fisher_w=0.1
        n_jobs=16
        Exp,Opn,Target,RE_TGlink=load_data_scNN(GRNdir,species)
        import warnings
        import time
        from tqdm import tqdm
        netall_s={}
        shapall_s={}
        #result=np.zeros([data_merge.shape[0],2])
        chrall=[RE_TGlink[0][i][0].split(':')[0] for i in range(RE_TGlink.shape[0])]
        RE_TGlink['chr']=chrall
        chrlist=RE_TGlink['chr'].unique()
        for jj in tqdm(range(len(chrlist))):
            chrtemp=chrlist[jj]
            RE_TGlink1=RE_TGlink[RE_TGlink['chr']==chrtemp]
            Lossall=np.zeros([RE_TGlink1.shape[0],100])
            for ii in  range(RE_TGlink1.shape[0]):
                warnings.filterwarnings("ignore")
                #res = Parallel(n_jobs=n_jobs)(delayed(sc_nn_NN)(ii,RE_TGlink_temp,Target,netall,Exp,Opn,l1_lambda,activef)  for ii in tqdm(range(RE_TGlink.shape[0]))
                RE_TGlink_temp=RE_TGlink1.values[ii,:]
                res=sc_nn_NN(ii,RE_TGlink_temp,Target,Exp,Opn,l1_lambda,activef)
                warnings.resetwarnings()
                netall_s[ii]=res[0]
                shapall_s[ii]=res[1]
                Lossall[ii,:]=res[2].T   
            torch.save(netall_s,outdir+chrtemp+'_net.pt')
            torch.save(shapall_s,outdir+chrtemp+'_shap.pt')
            Lossall=pd.DataFrame(Lossall)
            Lossall.index=RE_TGlink1['gene'].values
            Lossall.to_csv(outdir+chrtemp+'_Loss.txt',sep='\t') 
        RE_TGlink.to_csv(outdir+'RE_TGlink.txt',sep='\t',index=None)


def get_TSS_ensembl(genome_short,gtf_file,GRNdir):
    import pyensembl
    import subprocess
    from pyensembl import Genome
    ensembl = Genome(
    reference_name=genome_short,
    annotation_name="My_annotation",
    gtf_path_or_url=gtf_file)
    ensembl.index()
    genes = ensembl.genes()
# Retrieve TSS positions for each gene and store them in a list
    tss_positions = []
    strand=[]
    chrom=[]
    genesymbol=[]
    for gene in genes:
        tss_positions.append(gene.transcripts[0].start)
        strand.append(gene.strand)
        chrom.append('chr'+gene.contig)
        genesymbol.append(gene.name)
    import pandas as pd
    Tssdf = pd.DataFrame({'chr': chrom, 'TSS': tss_positions, 'symbol': genesymbol,'strand': strand})
    Tssdf.to_csv(GRNdir+'TSS_'+genome_short+'.txt',sep='\t',index=None,header=0)

In [18]:
import pandas as pd
outdir  = 'LINGER_output/'
GRNdir  = 'LINGER_data/data_bulk/'

In [ ]:
#import LingerGRN.LINGER_tr as LINGER_tr
training(GRNdir,'LINGER',outdir,'ReLU','Human')

## 6. Cell population GRN

### 6.1 TF-RE

`TF_RE_binding` calls `TF_RE_LINGER_chr`for each chromosome.

It extracts the first layer weights of the trained network — this is a matrix of shape (64, n_inputs) where n_inputs = n_TFs + n_REs for this gene. The first n_TFs columns correspond to TF inputs, the remaining n_RE columns correspond to RE inputs.

This gives a score for each TF-RE pair — how similarly the network weighted them in its first layer. High cosine similarity means the network learned that this TF and this RE jointly contribute to predicting the gene's expression, implying the TF binds at that RE.

- `chr{N}_cell_population_TF_RE_binding.txt` : per chr TF-RE binding matrix
- `cell_population_TF_RE_binding.txt  (peaks × TFs)` : concat all chr

In [ ]:
import LingerGRN.LL_net as LL_net
LL_net.TF_RE_binding(GRNdir,adata_RNA,adata_ATAC,genome,method,outdir)

### 6.2 RE-TG

`cis_reg` calls `cis_shap` for each chromosome, basically recovers shap values for all (valid) genes on that chr.

The SHAP values have shape `metacells × (n_TFs + n_REs)` — one value per input feature per metacell. Taking the mean absolute value across metacells gives one importance score per input feature. Then only the RE portion is kept — the last `n_RE` columns — giving one score per nearby peak for how much it contributed to predicting this gene's expression.

These RE importance scores directly become the cis-regulatory scores — no multiplication with distance decay or prior scores as in baseline.


In [ ]:
LL_net.cis_reg(GRNdir,adata_RNA,adata_ATAC,genome,method,outdir)

In [ ]:
re_tg = pd.read_csv("LINGER_output/cell_population_cis_regulatory.txt", sep="\t")
re_tg.shape

In [ ]:
re_tg.head(2)

### 6.3 TF-TG
`trans_reg` calls `trans_shap` for each chromosome, basically recovers shap values for all (valid) genes on that chr.

Same SHAP values as 6.2, same mean absolute operation, just the opposite slice — the first `n_TF` columns instead of the last `n_RE` columns.


In [ ]:
LL_net.trans_reg(GRNdir,method,outdir,genome)

In [ ]:
tf_tg = pd.read_csv("LINGER_output/cell_population_trans_regulatory.txt", sep="\t")
tf_tg.shape

In [ ]:
tf_tg.head(2)   # TF x TG

## Conclusion
All the three steps are independent. They all require those files : 
```
shap_{chr}.pt       — from training
net_{chr}.pt        — from training (TF_RE_binding only)
data_merge.txt      — from training
index.txt           — from preprocess
result_{chr}.txt    — from training
TFName.txt          — from preprocess
data/Peaks.txt      — from pseudobulk
Region_overlap_{chr}.bed  — from preprocess
```

```
preprocess ──→ training ──→ TF_RE_binding  → cell_population_TF_RE_binding.txt
                        ├── cis_reg        → cell_population_cis_regulatory.txt
                        └── trans_reg      → cell_population_trans_regulatory.txt
```